# Convolutional Neural Networks from Scratch

**Learning objectives**
- Understand convolution, stride, padding, and pooling with equations
- Implement a tiny CNN in NumPy (forward + backprop)
- See how learned filters detect oriented edges
- Compare with PyTorch's `nn.Conv2d`

Run cells top-to-bottom. Constants are grouped near the top so you can experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Hyperparameters (tweak these) ---
IMG_SIZE = 8
N_TRAIN = 800
N_TEST = 200
CONV_FILTERS = 4
KERNEL = 3
POOL = 2
HIDDEN = 16
LEARNING_RATE = 0.05
EPOCHS = 40
BATCH_SIZE = 32

## 1. Problem setup

CNNs exploit **spatial locality** and **weight sharing**: the same small filter scans the whole image.

We synthesize 8×8 binary images of **horizontal** vs **vertical** bars — a toy task where edge-detecting filters are the natural features.

In [ ]:
def make_bar_dataset(n, img_size=IMG_SIZE, rng=None):
    # Horizontal (y=0) vs vertical (y=1) thick bars with noise.
    rng = rng or np.random.default_rng(0)
    X = np.zeros((n, 1, img_size, img_size), dtype=np.float64)
    y = rng.integers(0, 2, size=n)
    for i in range(n):
        thickness = rng.integers(1, 3)
        if y[i] == 0:  # horizontal
            row = rng.integers(1, img_size - thickness)
            X[i, 0, row:row + thickness, :] = 1.0
        else:  # vertical
            col = rng.integers(1, img_size - thickness)
            X[i, 0, :, col:col + thickness] = 1.0
        X[i] += rng.normal(0, 0.05, size=X[i].shape)
        X[i] = np.clip(X[i], 0, 1)
    return X, y


X_train, y_train = make_bar_dataset(N_TRAIN, rng=np.random.default_rng(0))
X_test, y_test = make_bar_dataset(N_TEST, rng=np.random.default_rng(1))

fig, axes = plt.subplots(2, 6, figsize=(10, 3.5))
for ax, img, label in zip(axes[0], X_train[y_train == 0][:6], [0] * 6):
    ax.imshow(img[0], cmap='gray', vmin=0, vmax=1)
    ax.set_title('horiz')
    ax.axis('off')
for ax, img in zip(axes[1], X_train[y_train == 1][:6]):
    ax.imshow(img[0], cmap='gray', vmin=0, vmax=1)
    ax.set_title('vert')
    ax.axis('off')
plt.suptitle('Synthetic bar images')
plt.tight_layout()
plt.show()
print(f'X_train shape: {X_train.shape}  (N, C, H, W)')

## 2. Convolution — the core idea

A 2D convolution of input $X$ (one channel) with filter $K$ of size $k \times k$:

$$
(X * K)_{i,j} = \sum_{u=0}^{k-1} \sum_{v=0}^{k-1} X_{i+u,\, j+v}\, K_{u,v} + b
$$

**Output spatial size** (valid / no padding, stride $s$):

$$
H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} - k}{s} \right\rfloor + 1
$$

With **same padding** $p = \lfloor k/2 \rfloor$, $H_{\text{out}} = H_{\text{in}}$ for $s=1$.

**Learning note:** Dense layers on flattened pixels need a separate weight per pixel location. Convolution reuses one $k\times k$ kernel everywhere — far fewer parameters, and translation-equivariant features.

In [ ]:
def conv2d_forward(X, W, b, stride=1, padding=0):
    # X: (N,C,H,W), W: (F,C,k,k), b: (F,) -> out (N,F,Ho,Wo) + cache.
    N, C, H, W_in = X.shape
    F, _, k, _ = W.shape
    if padding:
        X_pad = np.pad(X, ((0, 0), (0, 0), (padding, padding), (padding, padding)))
    else:
        X_pad = X
    Hp, Wp = X_pad.shape[2], X_pad.shape[3]
    Ho = (Hp - k) // stride + 1
    Wo = (Wp - k) // stride + 1
    out = np.zeros((N, F, Ho, Wo))
    for i in range(Ho):
        for j in range(Wo):
            hs, ws = i * stride, j * stride
            patch = X_pad[:, :, hs:hs + k, ws:ws + k]  # (N,C,k,k)
            # Einstein: sum over C,k,k for each filter
            out[:, :, i, j] = np.tensordot(patch, W, axes=([1, 2, 3], [1, 2, 3])) + b
    cache = (X, W, b, stride, padding, X_pad)
    return out, cache


# Demo: a hand-crafted vertical-edge filter
demo = X_train[y_train == 1][:1]  # vertical bar
K_vert = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=np.float64).reshape(1, 1, 3, 3)
out, _ = conv2d_forward(demo, K_vert, np.zeros(1), padding=1)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(demo[0, 0], cmap='gray'); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(out[0, 0], cmap='coolwarm'); axes[1].set_title('vertical edge response'); axes[1].axis('off')
plt.tight_layout(); plt.show()

### Pause & Reflect — Convolution

1. Why does the same $3\times3$ kernel detect a vertical edge **anywhere** in the image?
2. If you rotate a vertical-edge kernel 90°, what should it detect?
3. How many parameters does one $3\times3$ filter on 1 input channel have vs a dense layer mapping $8\times8\to 6\times6$?

### Discussion — Convolution

1. **Weight sharing**: The kernel is slid; response depends on local pattern, not absolute $(i,j)$.
2. **Rotation**: A 90° rotation yields a horizontal-edge detector.
3. **Params**: Conv = $3\cdot3 + 1$ bias $= 10$. Dense $64\to36$ = $64\cdot36 + 36 = 2340$ — two orders of magnitude more, and no spatial structure.

## 3. Pooling and architecture

**Max-pool** (window $p\times p$, stride $p$) downsamples and keeps the strongest activation:

$$
\text{pool}(Z)_{i,j} = \max_{u,v \in \{0,\ldots,p-1\}} Z_{pi+u,\, pj+v}
$$

Our tiny CNN:

```
Input (1×8×8)
  → Conv (4 filters, 3×3, pad=1) + ReLU     → 4×8×8
  → MaxPool 2×2                             → 4×4×4
  → Flatten                                 → 64
  → Dense → ReLU → Dense → Softmax          → 2 classes
```

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def maxpool_forward(X, size=2, stride=2):
    N, C, H, W = X.shape
    Ho, Wo = (H - size) // stride + 1, (W - size) // stride + 1
    out = np.zeros((N, C, Ho, Wo))
    switches = np.zeros_like(X)  # for backprop
    for i in range(Ho):
        for j in range(Wo):
            hs, ws = i * stride, j * stride
            window = X[:, :, hs:hs + size, ws:ws + size]
            flat = window.reshape(N, C, -1)
            idx = np.argmax(flat, axis=2)
            out[:, :, i, j] = flat[np.arange(N)[:, None], np.arange(C)[None, :], idx]
            # mark switch
            for n in range(N):
                for c in range(C):
                    u, v = divmod(int(idx[n, c]), size)
                    switches[n, c, hs + u, ws + v] = 1
    return out, (X, switches, size, stride)


def one_hot(y, k=2):
    Y = np.zeros((len(y), k))
    Y[np.arange(len(y)), y] = 1.0
    return Y


def init_cnn(rng=np.random.default_rng(42)):
    # He init
    W_conv = rng.normal(0, np.sqrt(2 / (1 * KERNEL * KERNEL)),
                        size=(CONV_FILTERS, 1, KERNEL, KERNEL))
    b_conv = np.zeros(CONV_FILTERS)
    flat = CONV_FILTERS * (IMG_SIZE // POOL) * (IMG_SIZE // POOL)
    W1 = rng.normal(0, np.sqrt(2 / flat), size=(flat, HIDDEN))
    b1 = np.zeros((1, HIDDEN))
    W2 = rng.normal(0, np.sqrt(2 / HIDDEN), size=(HIDDEN, 2))
    b2 = np.zeros((1, 2))
    return {'W_conv': W_conv, 'b_conv': b_conv, 'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}


params = init_cnn()
print({k: v.shape for k, v in params.items()})

## 4. Forward, loss, and backprop (sketch)

**Categorical cross-entropy** with softmax logits $Z$:

$$
L = -\frac{1}{N}\sum_{n}\sum_{c} Y_{nc}\log \hat{Y}_{nc}, \qquad
\frac{\partial L}{\partial Z} = \hat{Y} - Y
$$

**Conv backprop intuition:** $\partial L / \partial K$ is itself a correlation of the input patches with the upstream gradient — filters that fired on useful patterns get reinforced.

We implement the full chain in NumPy below.

In [ ]:
def conv2d_backward(dout, cache):
    X, W, b, stride, padding, X_pad = cache
    N, C, H, W_in = X.shape
    F, _, k, _ = W.shape
    _, _, Ho, Wo = dout.shape
    dX_pad = np.zeros_like(X_pad)
    dW = np.zeros_like(W)
    db = dout.sum(axis=(0, 2, 3))
    for i in range(Ho):
        for j in range(Wo):
            hs, ws = i * stride, j * stride
            patch = X_pad[:, :, hs:hs + k, ws:ws + k]  # (N,C,k,k)
            # dW[f] += sum_n dout[n,f,i,j] * patch[n]
            dW += np.tensordot(dout[:, :, i, j], patch, axes=([0], [0]))
            # dX_pad += dout[n,f] * W[f]
            dX_pad[:, :, hs:hs + k, ws:ws + k] += np.tensordot(
                dout[:, :, i, j], W, axes=([1], [0])
            )
    if padding:
        dX = dX_pad[:, :, padding:-padding, padding:-padding]
    else:
        dX = dX_pad
    return dX, dW, db


def maxpool_backward(dout, cache):
    X, switches, size, stride = cache
    dX = np.zeros_like(X)
    N, C, Ho, Wo = dout.shape
    for i in range(Ho):
        for j in range(Wo):
            hs, ws = i * stride, j * stride
            region = switches[:, :, hs:hs + size, ws:ws + size]
            dX[:, :, hs:hs + size, ws:ws + size] += (
                region * dout[:, :, i, j][:, :, None, None]
            )
    return dX


def forward_cnn(X, params):
    z_conv, conv_cache = conv2d_forward(X, params['W_conv'], params['b_conv'], padding=1)
    a_conv = relu(z_conv)
    pooled, pool_cache = maxpool_forward(a_conv, size=POOL, stride=POOL)
    flat = pooled.reshape(X.shape[0], -1)
    z1 = flat @ params['W1'] + params['b1']
    a1 = relu(z1)
    z2 = a1 @ params['W2'] + params['b2']
    probs = softmax(z2)
    cache = (conv_cache, z_conv, a_conv, pool_cache, pooled, flat, z1, a1, z2, probs)
    return probs, cache


def loss_and_acc(probs, y):
    Y = one_hot(y)
    eps = 1e-8
    loss = float(-np.mean(np.sum(Y * np.log(np.clip(probs, eps, 1)), axis=1)))
    acc = float(np.mean(np.argmax(probs, axis=1) == y))
    return loss, acc


def backward_cnn(X, y, params, cache):
    conv_cache, z_conv, a_conv, pool_cache, pooled, flat, z1, a1, z2, probs = cache
    N = X.shape[0]
    Y = one_hot(y)
    dz2 = (probs - Y) / N
    dW2 = a1.T @ dz2
    db2 = dz2.sum(axis=0, keepdims=True)
    da1 = dz2 @ params['W2'].T
    dz1 = da1 * (z1 > 0)
    dW1 = flat.T @ dz1
    db1 = dz1.sum(axis=0, keepdims=True)
    dflat = dz1 @ params['W1'].T
    dpooled = dflat.reshape(pooled.shape)
    da_conv = maxpool_backward(dpooled, pool_cache)
    dz_conv = da_conv * (z_conv > 0)
    dX, dW_conv, db_conv = conv2d_backward(dz_conv, conv_cache)
    grads = {
        'W_conv': dW_conv, 'b_conv': db_conv,
        'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2,
    }
    return grads


probs0, _ = forward_cnn(X_train[:4], params)
print('init probs (4 samples):\n', np.round(probs0, 3))
print('init loss/acc:', loss_and_acc(probs0, y_train[:4]))

## 5. Training loop

In [ ]:
def train_cnn(X, y, X_te, y_te, epochs=EPOCHS, lr=LEARNING_RATE, batch=BATCH_SIZE):
    params = init_cnn()
    rng = np.random.default_rng(0)
    history = {'loss': [], 'train_acc': [], 'test_acc': []}
    n = len(y)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        for start in range(0, n, batch):
            bi = idx[start:start + batch]
            probs, cache = forward_cnn(X[bi], params)
            grads = backward_cnn(X[bi], y[bi], params, cache)
            for k in params:
                params[k] -= lr * grads[k]
        tr_probs, _ = forward_cnn(X, params)
        te_probs, _ = forward_cnn(X_te, params)
        tr_loss, tr_acc = loss_and_acc(tr_probs, y)
        _, te_acc = loss_and_acc(te_probs, y_te)
        history['loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'epoch {epoch+1:3d}  loss={tr_loss:.3f}  train={tr_acc*100:.1f}%  test={te_acc*100:.1f}%')
    return params, history


params, hist = train_cnn(X_train, y_train, X_test, y_test)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(hist['loss'], color='steelblue', label='loss')
ax2.plot([a * 100 for a in hist['test_acc']], color='coral', label='test acc %')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax2.set_ylabel('test accuracy %')
ax1.set_title('CNN training')
ax1.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Visualize learned convolutional filters
fig, axes = plt.subplots(1, CONV_FILTERS, figsize=(8, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(params['W_conv'][i, 0], cmap='coolwarm')
    ax.set_title(f'filter {i}')
    ax.axis('off')
plt.suptitle('Learned 3×3 filters (often resemble oriented edge detectors)')
plt.tight_layout(); plt.show()

### Pause & Reflect — CNN training

1. Why is max-pooling after ReLU, not before?
2. If you set `CONV_FILTERS = 1`, what failure mode might you see?
3. Why pad the convolution to keep $8\times8$ feature maps before pooling?

### Discussion — CNN training

1. **Order**: ReLU zeros negatives; pooling then selects the strongest positive response. Pooling before ReLU can mix signs awkwardly and is less common for this tiny net.
2. **One filter**: May only capture one orientation well — accuracy on the other class can stall.
3. **Padding**: Preserves spatial resolution so the $2\times2$ pool lands on a clean $4\times4$ grid without chopping the border.

## 6. PyTorch comparison

`nn.Conv2d` + `nn.MaxPool2d` implement the same ops with autograd.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, CONV_FILTERS, KERNEL, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(POOL),
            nn.Flatten(),
            nn.Linear(CONV_FILTERS * (IMG_SIZE // POOL) ** 2, HIDDEN),
            nn.ReLU(),
            nn.Linear(HIDDEN, 2),
        )
    def forward(self, x):
        return self.net(x)

model = TinyCNN()
opt = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
crit = nn.CrossEntropyLoss()

loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                  torch.tensor(y_train, dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=True,
)
X_te_t = torch.tensor(X_test, dtype=torch.float32)
y_te_t = torch.tensor(y_test, dtype=torch.long)

torch_accs = []
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t).argmax(1)
        torch_accs.append((pred == y_te_t).float().mean().item())
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'PyTorch epoch {epoch+1:3d}  test={torch_accs[-1]*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([a * 100 for a in hist['test_acc']], label='NumPy CNN', linewidth=2)
ax.plot([a * 100 for a in torch_accs], label='PyTorch CNN', linewidth=2)
ax.set_xlabel('epoch'); ax.set_ylabel('test accuracy %')
ax.set_title('NumPy vs PyTorch'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

| Idea | Takeaway |
|------|----------|
| Convolution | Local weighted sum + shared kernel → edge/texture detectors |
| Pooling | Spatial downsampling; keeps strongest activation |
| CNN stack | Conv → nonlinearity → pool → repeat → dense classifier |
| Why CNNs | Fewer params than dense-on-pixels; translation-friendly features |

**Next:** RNNs for sequences where order and history matter.